# Phân Tích Khám Phá Dữ Liệu Đa Phương Thức (Multimodal EDA)
## Video Transcript ↔ Keyframe Semantic Correlation Analysis

**Mục tiêu:** Phân tích mối tương quan ngữ nghĩa giữa Transcript và Keyframe
**Kiến trúc embedding:**
- Visual: PE-Core-bigG-14-448 → 1280 chiều
- Transcript: multilingual-e5-small → 384 chiều

**Cách tiếp cận:** Do hai embedding khác không gian (1280 vs 384), ta sử dụng similarity score
được tính gián tiếp qua cross-modal projection hoặc query-based matching.
Trong notebook này, ta mô phỏng similarity score dựa trên phân phối thực tế.

## 1. Import Thư Viện & Cấu Hình

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import random
import warnings

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')

# Cấu hình hiển thị
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'

# Seed cho reproducibility
np.random.seed(42)
random.seed(42)

print('Thư viện đã sẵn sàng.')

## 2. Tạo Dữ Liệu Mock — Mô Phỏng Schema Production

In [ ]:
# ─── Tham số cấu hình ────────────────────────────────────────────────────

NUM_FRAMES = 50
NUM_TRANSCRIPTS = 20
NUM_VIDEOS = 3
FPS = 25.0
VISUAL_DIM = 1280
TRANSCRIPT_DIM = 384
TOLERANCE_MS = 3000  # Dung sai ±3s khi map transcript ↔ frame

# ─── 14 chủ đề tiếng Việt (theo schema production) ──────────────────────

VIETNAMESE_TOPICS = [
    'Ẩm thực', 'Công nghệ', 'Du lịch', 'Thể thao',
    'Giáo dục', 'Kinh tế', 'Sức khỏe', 'Giải trí',
    'Thời sự', 'Văn hóa', 'Đời sống', 'Môi trường',
    'Giao thông', 'Pháp luật'
]

# ─── Từ khóa đặc trưng cho từng chủ đề (để sinh transcript có ý nghĩa) ──

TOPIC_KEYWORDS = {
    'Ẩm thực':   ['nấu ăn', 'món ăn', 'nguyên liệu', 'phở', 'bún', 'hương vị', 'đầu bếp'],
    'Công nghệ':  ['trí tuệ nhân tạo', 'phần mềm', 'máy tính', 'lập trình', 'AI', 'robot', 'chip'],
    'Du lịch':    ['du lịch', 'khách sạn', 'bãi biển', 'check-in', 'tour', 'điểm đến', 'khám phá'],
    'Thể thao':   ['bóng đá', 'vận động viên', 'giải đấu', 'sân vận động', 'bàn thắng', 'chạy'],
    'Giáo dục':   ['bài giảng', 'học sinh', 'sinh viên', 'thi cử', 'trường học', 'đào tạo'],
    'Kinh tế':    ['thị trường', 'doanh nghiệp', 'đầu tư', 'tăng trưởng', 'giá cả', 'lợi nhuận'],
    'Sức khỏe':   ['bác sĩ', 'bệnh viện', 'điều trị', 'vaccine', 'dinh dưỡng', 'sức khỏe'],
    'Giải trí':   ['phim ảnh', 'âm nhạc', 'ca sĩ', 'gameshow', 'sân khấu', 'nghệ sĩ'],
    'Thời sự':    ['tin tức', 'chính phủ', 'hội nghị', 'phát biểu', 'lãnh đạo', 'thời sự'],
    'Văn hóa':    ['truyền thống', 'lễ hội', 'nghệ thuật', 'bảo tàng', 'di sản', 'áo dài'],
    'Đời sống':   ['gia đình', 'mua sắm', 'sinh hoạt', 'hàng ngày', 'cộng đồng', 'chợ'],
    'Môi trường': ['khí hậu', 'ô nhiễm', 'rừng', 'bảo vệ môi trường', 'rác thải', 'xanh'],
    'Giao thông': ['đường xá', 'kẹt xe', 'cầu đường', 'xe buýt', 'giao thông', 'tàu điện'],
    'Pháp luật':  ['tòa án', 'luật', 'quy định', 'xét xử', 'công an', 'pháp luật'],
}

print(f'Cấu hình: {NUM_FRAMES} frames × {NUM_TRANSCRIPTS} transcripts × {NUM_VIDEOS} videos')

In [ ]:
# ─── Sinh video metadata ─────────────────────────────────────────────────

VIDEO_IDS = ['L01_V001', 'L01_V002', 'L01_V003']
YOUTUBE_IDS = ['dQw4w9WgXcQ', 'jNQXAC9IVRw', '9bZkp7q19f0']

videos = {
    vid: {
        'video_id': vid,
        'title': f'Video mẫu {i+1} — {random.choice(VIETNAMESE_TOPICS)}',
        'youtube_id': YOUTUBE_IDS[i],
        'fps': FPS,
        'duration_ms': 120_000,  # 2 phút
        'frame_count': 3000,
        'genre': random.choice(VIETNAMESE_TOPICS),
    }
    for i, vid in enumerate(VIDEO_IDS)
}

# ─── Sinh frames ─────────────────────────────────────────────────────────

frames = []
for i in range(NUM_FRAMES):
    vid = VIDEO_IDS[i % NUM_VIDEOS]
    frame_num = random.randint(0, videos[vid]['frame_count'] - 1)
    ts_ms = int(frame_num / FPS * 1000)
    frames.append({
        'frame_id': f"{vid}_{frame_num:06d}",
        'video_id': vid,
        'frame_number': frame_num,
        'timestamp_ms': ts_ms,
        'visual_embedding': np.random.randn(VISUAL_DIM).astype(np.float32),
    })

df_frames = pd.DataFrame(frames).drop(columns=['visual_embedding'])
print(f'Frames: {len(frames)}')
df_frames.head(8)

In [ ]:
# ─── Sinh transcripts — mỗi transcript là một khoảng thời gian có nội dung ─

def _random_transcript(video_id, idx):
    """Sinh một transcript chunk với topic và từ khóa đặc trưng."""
    topic = random.choice(VIETNAMESE_TOPICS)
    keywords = TOPIC_KEYWORDS[topic]
    num_words = random.randint(8, 40)
    words = [random.choice(keywords) for _ in range(min(num_words, len(keywords) * 3))]
    # Chèn thêm từ nối để câu tự nhiên hơn
    fillers = ['là', 'và', 'của', 'trong', 'có', 'được', 'với', 'cho', 'này', 'rất']
    sentence_words = []
    for w in words:
        if random.random() < 0.4:
            sentence_words.append(random.choice(fillers))
        sentence_words.append(w)
    text = ' '.join(sentence_words)

    segment_len_ms = random.randint(3_000, 20_000)  # 3–20s mỗi đoạn
    start_ms = random.randint(0, 120_000 - segment_len_ms)
    end_ms = start_ms + segment_len_ms

    return {
        'video_id': video_id,
        'start_time_ms': start_ms,
        'end_time_ms': end_ms,
        'text': text,
        'topic': topic,
        'transcript_embedding': np.random.randn(TRANSCRIPT_DIM).astype(np.float32),
    }

transcripts = []
for i in range(NUM_TRANSCRIPTS):
    vid = VIDEO_IDS[i % NUM_VIDEOS]
    transcripts.append(_random_transcript(vid, i))

df_transcripts = pd.DataFrame(transcripts).drop(columns=['transcript_embedding'])
df_transcripts['word_count'] = df_transcripts['text'].apply(lambda t: len(t.split()))
print(f'Transcripts: {len(transcripts)}')
df_transcripts.head(10)

## 3. Temporal Mapping — Ghép Frame với Transcript Theo Thời Gian

**Nguyên lý:** Một keyframe được coi là thuộc về một transcript chunk nếu timestamp
của nó nằm trong khoảng `[start_time_ms, end_time_ms]` (có thể mở rộng với `tolerance_ms`).
Đây là bước quan trọng để xác định "frame này đang minh họa cho đoạn hội thoại nào".

In [ ]:
def temporal_map(frames, transcripts, tolerance_ms=TOLERANCE_MS):
    """
    Ghép mỗi frame với transcript chunk dựa trên thời gian.

    Điều kiện khớp:
      (start_time_ms - tolerance) <= timestamp_ms <= (end_time_ms + tolerance)

    
    Returns:
        list[dict]: Mỗi phần tử là một cặp frame-transcript đã khớp,
                    kèm theo gap_ms (khoảng cách đến tâm transcript).
    """
    matches = []

    for f in frames:
        vid = f['video_id']
        ts = f['timestamp_ms']

        for t in transcripts:
            if t['video_id'] != vid:
                continue  # Chỉ xét transcript cùng video

            start = t['start_time_ms'] - tolerance_ms
            end = t['end_time_ms'] + tolerance_ms

            if start <= ts <= end:
                center_ms = (t['start_time_ms'] + t['end_time_ms']) / 2
                gap_ms = abs(ts - center_ms)

                matches.append({
                    **f,
                    **{f'transcript_{k}': v for k, v in t.items()},
                    'gap_ms': gap_ms,
                    'word_count': len(t['text'].split()),
                })
                break  # Mỗi frame chỉ khớp với transcript đầu tiên thỏa mãn

    return matches

# Thực hiện temporal mapping
matched_pairs = temporal_map(frames, transcripts)
df_matched = pd.DataFrame(matched_pairs)
df_matched.rename(columns={
    'transcript_topic': 'topic',
    'transcript_text': 'text',
    'transcript_start_time_ms': 'start_time_ms',
    'transcript_end_time_ms': 'end_time_ms',
}, inplace=True)
print(f'Cặp frame-transcript khớp thời gian: {len(df_matched)} / {len(frames)} frames')
print(f'Tỷ lệ khớp: {len(df_matched) / len(frames) * 100:.1f}%')
df_matched.head(8)

### 3.1. Phân tích tỷ lệ khớp theo video

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Biểu đồ tỷ lệ frame được map / không được map
mapped_ids = {m['frame_id'] for m in matched_pairs}
df_frames['mapped'] = df_frames['frame_id'].isin(mapped_ids)
map_counts = df_frames.groupby(['video_id', 'mapped']).size().unstack(fill_value=0)

map_counts.plot(kind='bar', stacked=True, ax=axes[0],
                color=['#e74c3c', '#2ecc71'], edgecolor='white')
axes[0].set_title('Số Frame Được Map vs Không Map Theo Video', fontweight='bold')
axes[0].set_xlabel('Video ID')
axes[0].set_ylabel('Số lượng Frame')
axes[0].legend(['Không map', 'Đã map'])
axes[0].tick_params(axis='x', rotation=0)

# Phân phối gap_ms — kiểm tra chất lượng temporal mapping
axes[1].hist(df_matched['gap_ms'], bins=30, color='#3498db', edgecolor='white', alpha=0.85)
axes[1].axvline(df_matched['gap_ms'].median(), color='red', linestyle='--',
                linewidth=2, label=f'Median = {df_matched["gap_ms"].median():.0f} ms')
axes[1].set_title('Phân Phối Khoảng Cách Thời Gian (gap_ms)\nTừ Frame đến Tâm Transcript',
                   fontweight='bold')
axes[1].set_xlabel('gap_ms (milliseconds)')
axes[1].set_ylabel('Số cặp')
axes[1].legend()

plt.tight_layout()
plt.show()

# ─── Insight ──────────────────────────────────────────────────────────────
# Gap càng nhỏ → frame càng gần tâm transcript → ngữ cảnh thời gian càng chặt chẽ.
# Gap lớn có thể báo hiệu frame nằm ở rìa transcript, khả năng lệch ngữ nghĩa cao hơn.

## 4. Sinh Similarity Score & Phân Loại Chất Lượng

**Nguyên lý:** Do visual embedding (1280d) và transcript embedding (384d) khác không gian,
similarity score trong production đến từ query-based matching (text query → PE-Core vector →
Milvus cosine search). Ở đây ta mô phỏng similarity score dựa trên phân phối thực tế:
- Tạo similarity giả lập có tương quan nghịch với `gap_ms` (frame càng gần tâm → score càng cao).
- Thêm noise để phản ánh thực tế không phải lúc nào temporal alignment cũng = semantic alignment.

In [ ]:
def generate_similarity_scores(df, score_noise_std=0.15):
    """
    Mô phỏng similarity score giữa transcript và frame.

    Công thức:
      base_score = 0.85 - 0.4 * (gap_ms / max_gap)  # gap càng nhỏ → score càng cao
      noise      = N(0, score_noise_std)
      score      = clip(base_score + noise, -1, 1)

    Score cao (~0.7+) → transcript mô tả tốt nội dung frame.
    Score thấp (~<0.3) → transcript không liên quan hoặc lệch ngữ cảnh.
    """
    df = df.copy()
    max_gap = df['gap_ms'].max() or 1.0

    base = 0.85 - 0.4 * (df['gap_ms'] / max_gap)
    noise = np.random.normal(0, score_noise_std, len(df))
    df['similarity_score'] = np.clip(base + noise, -1.0, 1.0)

    return df

df_matched = generate_similarity_scores(df_matched)
print(f'Similarity score: min={df_matched["similarity_score"].min():.4f}, '
      f'max={df_matched["similarity_score"].max():.4f}, '
      f'mean={df_matched["similarity_score"].mean():.4f}')
df_matched[['frame_id', 'video_id', 'timestamp_ms', 'topic', 'similarity_score']].head(8)

## 5. Threshold-Based Classification & Histogram

**Nguyên lý:** Chọn ngưỡng `threshold` để phân các cặp frame-transcript thành:
- **Good Match (Đại diện tốt):** similarity_score >= threshold
- **Bad Match (Đại diện kém):** similarity_score < threshold

Ngưỡng có thể điều chỉnh dựa trên yêu cầu chất lượng của hệ thống retrieval.

In [ ]:
THRESHOLD = 0.5  # Có thể điều chỉnh

df_matched['match_quality'] = np.where(
    df_matched['similarity_score'] >= THRESHOLD, 'Good Match', 'Bad Match'
)

good_count = (df_matched['match_quality'] == 'Good Match').sum()
bad_count = (df_matched['match_quality'] == 'Bad Match').sum()

print(f'Phân loại (threshold = {THRESHOLD}):')
print(f'  Good Match: {good_count} ({good_count / max(len(df_matched),1) * 100:.1f}%)')
print(f'  Bad Match:  {bad_count} ({bad_count / max(len(df_matched),1) * 100:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# ─── Histogram similarity score ───────────────────────────────────────────

ax = axes[0]
ax.hist(df_matched['similarity_score'], bins=25, color='#9b59b6', edgecolor='white', alpha=0.8)
ax.axvline(THRESHOLD, color='red', linestyle='--', linewidth=2.5,
           label=f'Threshold = {THRESHOLD}')

# Tô màu phân vùng
ymin, ymax = ax.get_ylim()
ax.fill_betweenx([0, ymax], -1, THRESHOLD, alpha=0.08, color='red', label='Vùng Bad Match')
ax.fill_betweenx([0, ymax], THRESHOLD, 1, alpha=0.08, color='green', label='Vùng Good Match')

ax.set_title(f'Phân Phối Similarity Score\n(Threshold = {THRESHOLD})', fontweight='bold', fontsize=13)
ax.set_xlabel('Cosine Similarity Score')
ax.set_ylabel('Số cặp frame-transcript')
ax.set_xlim(-1, 1)
ax.legend(loc='upper left')

# ─── Pie chart tỷ lệ Good/Bad ────────────────────────────────────────────

axes[1].pie([good_count, bad_count],
            labels=[f'Good Match\n({good_count})', f'Bad Match\n({bad_count})'],
            colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%',
            startangle=90, explode=(0.02, 0.02),
            textprops={'fontsize': 12, 'fontweight': 'bold'})
axes[1].set_title('Tỷ Lệ Phân Loại Match Quality', fontweight='bold', fontsize=13)

plt.tight_layout()
plt.show()

# ─── Insight ──────────────────────────────────────────────────────────────
# Ngưỡng 0.5 là lựa chọn cân bằng. Nếu muốn precision cao → tăng threshold.
# Nếu muốn recall cao (bắt nhiều frame hơn) → giảm threshold.
# Sự phân bố lệch phải cho thấy đa số cặp có tương quan ngữ nghĩa tích cực.

## 6. EDA Insights — Phân Tích Chuyên Sâu

### 6.1. Text Length vs. Similarity Score

**Câu hỏi nghiên cứu:** Transcript dài hơn (nhiều từ hơn) có tạo ra similarity score cao hơn không?
- Transcript dài → chứa nhiều thông tin → khả năng khớp với nội dung frame cao hơn.
- Nhưng transcript quá dài có thể bị nhiễu (nhiều thông tin không liên quan).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# ─── Scatter plot: Word Count vs Similarity ───────────────────────────────

ax = axes[0]
colors = df_matched['match_quality'].map({'Good Match': '#2ecc71', 'Bad Match': '#e74c3c'})
sc = ax.scatter(df_matched['word_count'], df_matched['similarity_score'],
                c=colors, alpha=0.7, s=80, edgecolors='white', linewidth=0.5)

# Đường hồi quy
from numpy.polynomial.polynomial import polyfit
x = df_matched['word_count']
y = df_matched['similarity_score']
b, m = polyfit(x, y, 1)
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, b + m * x_line, '--', color='#34495e', linewidth=2, alpha=0.7,
        label=f'y = {m:.4f}x + {b:.3f}')

ax.set_title('Độ Dài Transcript vs Similarity Score\n(Mỗi điểm = 1 cặp transcript–frame)',
             fontweight='bold')
ax.set_xlabel('Số từ trong Transcript')
ax.set_ylabel('Similarity Score')
ax.axhline(THRESHOLD, color='gray', linestyle=':', linewidth=1.5, alpha=0.5)
ax.legend()

# ─── Box plot: Word count theo match quality ──────────────────────────────

ax2 = axes[1]
bp = ax2.boxplot(
    [df_matched[df_matched['match_quality'] == 'Good Match']['word_count'],
     df_matched[df_matched['match_quality'] == 'Bad Match']['word_count']],
    tick_labels=['Good Match', 'Bad Match'],
    patch_artist=True,
    widths=0.4
)
bp['boxes'][0].set_facecolor('#2ecc71')
bp['boxes'][0].set_alpha(0.6)
bp['boxes'][1].set_facecolor('#e74c3c')
bp['boxes'][1].set_alpha(0.6)

ax2.set_title('Phân Phối Word Count Theo Match Quality', fontweight='bold')
ax2.set_ylabel('Số từ trong Transcript')

plt.tight_layout()
plt.show()

# ─── Insight ──────────────────────────────────────────────────────────────
# Nếu đường hồi quy đi lên → transcript dài hơn giúp matching tốt hơn.
# Nếu đường hồi quy phẳng hoặc đi xuống → độ dài không ảnh hưởng hoặc gây nhiễu.
# Box plot cho thấy sự khác biệt về độ dài trung bình giữa 2 nhóm.

### 6.2. Temporal Gap Analysis

**Câu hỏi nghiên cứu:** Khoảng cách thời gian giữa frame và tâm transcript (`gap_ms`)
có tương quan với similarity score không?

**Intuition:** Frame càng xa tâm transcript → khả năng nội dung hình ảnh lệch khỏi ngữ
cảnh hội thoại càng cao → similarity score thấp hơn.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

ax = axes[0]
sc2 = ax.scatter(df_matched['gap_ms'] / 1000, df_matched['similarity_score'],
                 c=colors, alpha=0.7, s=80, edgecolors='white', linewidth=0.5)
ax.set_title('Khoảng Cách Thời Gian vs Similarity Score', fontweight='bold')
ax.set_xlabel('Gap từ Frame đến Tâm Transcript (giây)')
ax.set_ylabel('Similarity Score')

# Hồi quy
x_gap = df_matched['gap_ms']
b2, m2 = polyfit(x_gap / 1000, y, 1)
x_line2 = np.linspace(x_gap.min(), x_gap.max(), 100)
ax.plot(x_line2 / 1000, b2 + m2 * (x_line2 / 1000), '--', color='#34495e', linewidth=2, alpha=0.7,
        label=f'y = {m2:.6f}x + {b2:.3f}')
ax.legend()

# ─── Pearson correlation tổng quan ────────────────────────────────────────

corr_columns = ['similarity_score', 'gap_ms', 'word_count', 'start_time_ms', 'end_time_ms']
corr_matrix = df_matched[corr_columns].corr()

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0,
            fmt='.3f', linewidths=0.5, ax=axes[1],
            xticklabels=['Sim Score', 'Gap (ms)', 'Word Count', 'Start (ms)', 'End (ms)'],
            yticklabels=['Sim Score', 'Gap (ms)', 'Word Count', 'Start (ms)', 'End (ms)'])
axes[1].set_title('Ma Trận Tương Quan\n(Cross-Modal Features)', fontweight='bold')

plt.tight_layout()
plt.show()

# ─── Insight ──────────────────────────────────────────────────────────────
# Tương quan âm giữa gap_ms và similarity_score → frame càng gần tâm transcript
# thì similarity càng cao (phù hợp với cách sinh dữ liệu mô phỏng).
# Trong thực tế, gap_ms có thể không tương quan mạnh nếu transcript bao quát
# nhiều chủ đề hoặc frame không có nội dung thị giác liên quan đến lời nói.

### 6.3. Phân Tích Từ Khóa — Word Frequency trong Good Match

**Câu hỏi nghiên cứu:** Những từ khóa nào xuất hiện nhiều nhất trong các transcript
có similarity cao với frame? Điều này giúp xác định chủ đề/chủ điểm nào có
khả năng đại diện tốt cho nội dung hình ảnh.

In [ ]:
import re
from collections import Counter

# ─── Tokenizer đơn giản cho tiếng Việt ────────────────────────────────────

STOP_WORDS = {'là', 'và', 'của', 'trong', 'có', 'được', 'với', 'cho', 'này',
              'rất', 'một', 'những', 'các', 'đã', 'sẽ', 'đang', 'không', 'để',
              'từ', 'theo', 'về', 'khi', 'nên', 'ra', 'vào', 'còn', 'nhưng'}

def tokenize_vietnamese(text):
    """Tách từ tiếng Việt đơn giản (theo khoảng trắng), bỏ stop words và ký tự ngắn."""
    cleaned = re.sub(r'[^\w\sà-ỹ]', '', text.lower())
    tokens = cleaned.split()
    return [t for t in tokens if t not in STOP_WORDS and len(t) >= 2]

# ─── Tần suất từ trong Good Match ─────────────────────────────────────────

good_texts = df_matched[df_matched['match_quality'] == 'Good Match']['text']
bad_texts = df_matched[df_matched['match_quality'] == 'Bad Match']['text']

good_tokens = []
for text in good_texts:
    good_tokens.extend(tokenize_vietnamese(text))

bad_tokens = []
for text in bad_texts:
    bad_tokens.extend(tokenize_vietnamese(text))

good_freq = Counter(good_tokens).most_common(20)
bad_freq = Counter(bad_tokens).most_common(20)

df_good_freq = pd.DataFrame(good_freq, columns=['Từ', 'Tần suất (Good)'])
df_bad_freq = pd.DataFrame(bad_freq, columns=['Từ', 'Tần suất (Bad)'])

print('─ Top 10 từ khóa trong GOOD MATCH ─')
display(df_good_freq.head(10))
print('\n─ Top 10 từ khóa trong BAD MATCH ─')
display(df_bad_freq.head(10))

In [ ]:
# ─── So sánh từ khóa nổi bật giữa Good và Bad ─────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

def plot_word_freq(ax, data, title, color):
    if not data:
        return
    words, freqs = zip(*data[:15])
    y_pos = range(len(words))
    ax.barh(y_pos, freqs, color=color, edgecolor='white', alpha=0.85)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(words)
    ax.invert_yaxis()
    ax.set_title(title, fontweight='bold', fontsize=13)
    ax.set_xlabel('Tần suất')

plot_word_freq(axes[0], good_freq, 'Top 15 Từ Khóa — GOOD Match', '#2ecc71')
plot_word_freq(axes[1], bad_freq, 'Top 15 Từ Khóa — BAD Match', '#e74c3c')

plt.tight_layout()
plt.show()

# ─── Overlap analysis ─────────────────────────────────────────────────────

good_words_set = set(w for w, _ in good_freq)
bad_words_set = set(w for w, _ in bad_freq)
shared = good_words_set & bad_words_set
good_only = good_words_set - bad_words_set
bad_only = bad_words_set - good_words_set

print(f'Từ chỉ xuất hiện trong Good Match: {len(good_only)} → {", ".join(list(good_only)[:10])}')
print(f'Từ chỉ xuất hiện trong Bad Match:  {len(bad_only)} → {", ".join(list(bad_only)[:10])}')
print(f'Từ chung cả hai nhóm:             {len(shared)} → {", ".join(list(shared)[:10])}')

print('\n💡 Insight: Từ chỉ xuất hiện ở Good Match có thể là đặc trưng cho nội dung')
print('   được frame minh họa tốt (ví dụ: từ mô tả hành động, đối tượng cụ thể).')
print('   Từ chỉ xuất hiện ở Bad Match có thể là từ trừu tượng, khó thể hiện bằng hình ảnh.')

### 6.4. Topic Distribution Analysis

**Câu hỏi nghiên cứu:** Chủ đề nào có tỷ lệ "Good Match" cao nhất?
Điều này cho thấy những chủ đề dễ dàng được minh họa bằng hình ảnh hơn những chủ đề khác.

In [ ]:
# ─── Tỷ lệ Good/Bad theo chủ đề ───────────────────────────────────────────

topic_quality = (
    df_matched.groupby(['topic', 'match_quality']).size()
    .unstack(fill_value=0)
)
# Sắp xếp theo tổng số cặp
topic_quality['total'] = topic_quality.sum(axis=1)
topic_quality = topic_quality.sort_values('total', ascending=False).drop(columns='total')

# Tính tỷ lệ %
topic_pct = topic_quality.div(topic_quality.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ─── Stacked bar chart ────────────────────────────────────────────────────

colors_map = ['#e74c3c', '#2ecc71']
topic_quality.plot(kind='barh', stacked=True, ax=axes[0], color=colors_map,
                   edgecolor='white', width=0.7)
axes[0].set_title('Số Lượng Good/Bad Match Theo Chủ Đề', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Số cặp frame-transcript')
axes[0].set_ylabel('')
axes[0].legend(loc='lower right')

# ─── Tỷ lệ % Good Match theo chủ đề ───────────────────────────────────────

if 'Good Match' in topic_pct.columns:
    good_ratio = topic_pct['Good Match'].sort_values(ascending=True)
    bars = axes[1].barh(good_ratio.index, good_ratio.values, color='#2ecc71',
                        edgecolor='white', alpha=0.85)
    axes[1].axvline(50, color='gray', linestyle='--', alpha=0.5, label='50%')
    axes[1].set_title('Tỷ Lệ % Good Match Theo Chủ Đề', fontweight='bold', fontsize=13)
    axes[1].set_xlabel('% Good Match')
    axes[1].set_ylabel('')
    axes[1].set_xlim(0, 100)
    axes[1].legend()

plt.tight_layout()
plt.show()

# ─── Bảng xếp hạng chủ đề theo tỷ lệ Good Match ───────────────────────────

if 'Good Match' in topic_pct.columns:
    ranking = topic_pct['Good Match'].sort_values(ascending=False)
    print('Bảng xếp hạng chủ đề theo % Good Match:')
    for rank, (topic, pct) in enumerate(ranking.items(), 1):
        bar = '█' * int(pct / 2)
        print(f'  {rank:2d}. {topic:<12s}  {pct:5.1f}%  {bar}')

print('\n💡 Insight: Chủ đề có % Good Match cao → nội dung dễ minh họa bằng hình ảnh')
print('   (ví dụ: Ẩm thực, Thể thao). Chủ đề có % thấp → nội dung trừu tượng hơn')
print('   (ví dụ: Pháp luật, Kinh tế), khó thể hiện qua một keyframe.')

### 6.5. Similarity Score Distribution Theo Chủ Đề

**Câu hỏi nghiên cứu:** Phân phối similarity score có khác biệt giữa các chủ đề không?
Chủ đề nào có điểm số đồng đều? Chủ đề nào có biến động lớn?

In [ ]:
# ─── Violin plot similarity score theo chủ đề ─────────────────────────────

fig, ax = plt.subplots(figsize=(15, 7))

# Sắp xếp theo median score giảm dần
order = df_matched.groupby('topic')['similarity_score'].median().sort_values(ascending=False).index

sns.violinplot(
    data=df_matched, x='topic', y='similarity_score',
    order=order, palette='viridis', inner='quartile',
    ax=ax, linewidth=1, alpha=0.9
)

ax.axhline(THRESHOLD, color='red', linestyle='--', linewidth=2, alpha=0.7,
           label=f'Threshold = {THRESHOLD}')
ax.set_title('Phân Phối Similarity Score Theo Chủ Đề\n(Violin Plot — Median + Quartiles)',
             fontweight='bold', fontsize=14)
ax.set_xlabel('')
ax.set_ylabel('Similarity Score')
ax.tick_params(axis='x', rotation=45)
ax.legend()

plt.tight_layout()
plt.show()

# ─── Bảng thống kê ────────────────────────────────────────────────────────

topic_stats = df_matched.groupby('topic')['similarity_score'].agg(['mean', 'median', 'std', 'count'])
topic_stats = topic_stats.sort_values('median', ascending=False).round(4)
print('Thống kê Similarity Score theo Chủ Đề:')
display(topic_stats)

print('\n💡 Insight:')
print('  - Median cao + std thấp → chủ đề có chất lượng matching ổn định, dễ dự đoán.')
print('  - Median thấp + std cao → chủ đề khó matching, cần cải thiện embedding hoặc fusion strategy.')
print('  - Với dữ liệu mô phỏng, phân phối mang tính ngẫu nhiên. Với dữ liệu thật,')
print('    phân tích này sẽ tiết lộ điểm mạnh/yếu của PE-Core + multilingual-e5-small.')

## 7. Tổng Kết & Khuyến Nghị

### Tóm tắt các insights từ EDA

| Phân tích | Insight chính | Hành động đề xuất |
|---|---|---|
| **Temporal mapping** | Tỷ lệ frame không khớp transcript có thể cao nếu coverage thấp | Tăng mật độ transcript hoặc mở rộng tolerance |
| **Similarity histogram** | Phân phối lệch phải → đa số cặp có tương quan dương | Chọn threshold dựa trên precision/recall trade-off |
| **Text length vs Score** | Transcript quá ngắn hoặc quá dài có thể giảm chất lượng | Chuẩn hóa độ dài transcript chunk (20–50 từ) |
| **Temporal gap vs Score** | Gap càng nhỏ → score càng cao | Ưu tiên frame gần tâm transcript trong fusion strategy |
| **Topic analysis** | Một số chủ đề (trừu tượng) khó matching hơn | Tăng trọng số visual cho chủ đề cụ thể, tăng text cho chủ đề trừu tượng |
| **Từ khóa Good/Bad** | Từ cụ thể → Good; từ trừu tượng → Bad | Cân nhắc cross-modal attention hoặc keyword boosting |

### Hướng cải thiện cho Production

1. **Fine-tune temporal tolerance** dựa trên phân phối gap_ms thực tế.
2. **Dynamic threshold**: Điều chỉnh `threshold` theo chủ đề thay vì dùng một giá trị cố định.
3. **Multi-modal fusion weight tuning**: Tăng trọng số cho transcript với chủ đề khó visual-matching.
4. **Transcript chunk optimization**: Đảm bảo mỗi chunk có độ dài phù hợp, không quá ngắn (<10 từ) hoặc quá dài (>60 từ).
5. **Cross-modal projection learning**: Train một projection matrix ánh xạ transcript embedding (384d) → visual space (1280d) để tính cosine similarity trực tiếp.

## 8. Appendix — Dữ Liệu Thống Kê Tổng Hợp

In [ ]:
# ─── Bảng tổng hợp dữ liệu ────────────────────────────────────────────────

print(f'{"="*70}')
print(f'  BÁO CÁO EDA ĐA PHƯƠNG THỨC — TỔNG HỢP')
print(f'{"="*70}')
print(f'  Tổng số video:          {NUM_VIDEOS}')
print(f'  Tổng số frames:         {NUM_FRAMES}')
print(f'  Tổng số transcripts:    {NUM_TRANSCRIPTS}')
print(f'  Số cặp khớp thời gian:  {len(df_matched)} ({len(df_matched)/max(NUM_FRAMES,1)*100:.1f}%)')
print(f'  Ngưỡng phân loại:       {THRESHOLD}')
print(f'  Good Match:             {good_count} ({good_count/max(len(df_matched),1)*100:.1f}%)')
print(f'  Bad Match:              {bad_count} ({bad_count/max(len(df_matched),1)*100:.1f}%)')
print(f'  Similarity mean ± std:  {df_matched["similarity_score"].mean():.4f} ± {df_matched["similarity_score"].std():.4f}')
print(f'  Similarity median:      {df_matched["similarity_score"].median():.4f}')
print(f'  Gap_ms trung bình:      {df_matched["gap_ms"].mean():.0f} ms')
print(f'  Số chủ đề xuất hiện:    {df_matched["topic"].nunique()} / {len(VIETNAMESE_TOPICS)}')
print(f'{"="*70}')

# ─── Top 5 cặp tốt nhất ───────────────────────────────────────────────────

print('\n🏆 Top 5 cặp frame-transcript tốt nhất:')
top5 = df_matched.nlargest(5, 'similarity_score')[
    ['frame_id', 'video_id', 'topic', 'similarity_score', 'text']
]
display(top5)

print('\n⚠️  Top 5 cặp frame-transcript kém nhất:')
bot5 = df_matched.nsmallest(5, 'similarity_score')[
    ['frame_id', 'video_id', 'topic', 'similarity_score', 'text']
]
display(bot5)